# Lezione 7. Approssimazione di funzioni e Deep Q-Networks (DQN)

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

Pensa a una scacchiera: ogni disposizione possibile dei pezzi è uno stato diverso, e il numero di stati totali supera di gran lunga la quantità di atomi dell'universo osservabile. Oppure pensa a un videogioco, dove ogni fotogramma di pixel sullo schermo è, in pratica, un nuovo stato. Fino alla lezione sei avevi un valore esatto per ogni coppia stato-azione in una tabella. Oggi quella tabella smette di essere praticabile, e la sostituisci con una rete neurale che generalizza.

## Sigle di questa lezione

- **RL**: Apprendimento per Rinforzo (Reinforcement Learning)
- **DQN**: Deep Q-Network, rete Q profonda
- **TD**: Differenza Temporale (Temporal Difference)

## Scaletta della lezione

1. Dalle tabelle alle funzioni: l'idea dell'approssimazione di funzioni
2. Perché unire Q-learning e una rete neurale, senza altro, è instabile
3. I due elementi che stabilizzano DQN: buffer di replay e rete target
4. Codice dal vivo: DQN completo su CartPole, con PyTorch e gymnasium
5. Punti chiave e cosa ti aspetta nella lezione otto


## 1. Dalle tabelle alle funzioni: l'approssimazione di funzioni

*Traccia della lezione: riprendi l'analogia degli scacchi della copertina. Chiedi alla classe quante volte pensa che un agente di Q-learning tabulare visiterebbe due volte esattamente lo stesso stato in un videogioco reale. La risposta —quasi mai— è la porta d'ingresso a tutto il tema di oggi.*

La tabella del Q-learning tabulare funziona a meraviglia nei problemi piccoli, ma si rompe appena il numero di stati esplode: non sta in memoria e, soprattutto, il tuo agente quasi mai torna a visitare esattamente lo stesso stato due volte. Senza una tabella che lo ricordi, non impara nulla di utile su uno stato che vede una sola volta.

Nell'RL un agente impara sempre una di queste tre cose: una politica (quale azione prendere in ogni stato), una funzione di valore (quanto vale ogni stato o ogni coppia stato-azione) oppure un modello (a quale stato ti porta ogni azione). Tutte e tre sono, in fondo, funzioni. L'**approssimazione di funzioni** consiste nel rappresentare quella funzione con un insieme di parametri regolabili, invece che con una tabella esatta:

`Q(s,a;θ) ≈ Q_vero(s,a)`

dove θ rappresenta questi parametri. L'obiettivo dell'addestramento è trovare i valori di θ che avvicinano il più possibile quell'approssimazione al valore reale.

Una precisazione breve, nel caso le reti neurali ti siano ancora un territorio nuovo: una rete neurale è, in fondo, una funzione con parametri regolabili (i pesi), organizzata in strati che trasformano l'input passo dopo passo fino a produrre un output. Addestrarla significa spostare quei pesi, poco a poco e in modo automatico, finché la funzione che calcola somiglia sempre di più alla funzione che vuoi imitare; nel tuo caso, il valore di ogni azione possibile in ogni stato.

**Perché vale la pena il cambiamento?** Per due motivi:

- **Generalizzazione**: un agente che approssima Q con una funzione può stimare il valore di uno stato che non ha mai visitato, appoggiandosi a stati simili che già conosce
- **Meno esigenze per passo**: non devi più ottimizzare la politica alla perfezione in ogni ciclo; basta migliorare l'approssimazione in modo incrementale perché la politica migliori con essa

Esistono vari tipi di approssimatori —combinazioni lineari di caratteristiche costruite a mano, alberi decisionali, vicini più prossimi— ma oggi dominano le reti neurali, perché imparano direttamente da segnali grezzi, come i pixel di uno schermo, senza che tu debba progettare alcuna caratteristica a mano. L'idea non è nuova: negli anni ottanta Gerald Tesauro combinò l'apprendimento per Differenza Temporale (TD) con una rete neurale per giocare a backgammon, e Chris Watkins formulò il Q-learning nel 1989. Ciò che è cambiato da allora è la potenza di calcolo disponibile.


## 2. Perché unire Q-learning e una rete neurale, senza altro, è instabile

*Traccia della lezione: presenta la versione ingenua alla lavagna prima di spiegare perché fallisce. Lascia che la classe intuisca il problema prima di darlo.*

La versione ingenua è allettante: sostituisci la tabella del Q-learning con una rete neurale che riceve lo stato e restituisce un valore per ogni azione, scegli l'azione con epsilon-greedy, calcoli l'errore tra ciò che la rete ha previsto e ciò che "avrebbe dovuto" prevedere, e aggiusti i pesi con la discesa del gradiente. Sulla carta funziona. In pratica, addestrata così, la rete tende a oscillare senza convergere oppure diverge del tutto, per due motivi.

**Primo motivo: i dati non sono indipendenti.** I passi consecutivi di un episodio sono molto correlati tra loro: lo stato al passo cento assomiglia moltissimo allo stato al passo 101. È la differenza tra addestrarsi con foto scelte a caso da migliaia di scene diverse e addestrarsi guardando, fotogramma per fotogramma, gli stessi secondi di un unico film: la rete si specializza in ciò che ha appena visto e dimentica ciò che ha visto prima.

**Secondo motivo: il bootstrapping sposta lo stesso obiettivo.** Nel Q-learning, l'obiettivo (*target*) che la tua rete insegue viene calcolato usando la rete stessa:

`obiettivo = r + γ · max_a' Q(s',a';θ)`

Ogni volta che aggiusti i pesi θ per avvicinarti a quell'obiettivo, l'obiettivo si sposta anch'esso, perché dipende proprio da quei pesi. È come cercare di colpire un bersaglio che si sposta ogni volta che spari, e che per di più si sposta proprio perché hai sparato. A ciò si aggiunge che la politica che decide quali dati raccoglie l'agente cambia insieme alla rete, quindi i dati di addestramento cambiano nello stesso momento in cui cambia la rete che stai cercando di addestrare.


## 3. I due elementi che stabilizzano DQN

*Traccia della lezione: presenta ciascun elemento come la risposta diretta a uno dei due motivi di instabilità del blocco precedente. Il buffer attacca la correlazione tra i passi; la rete target attacca il bootstrapping.*

DQN (Mnih e collaboratori, 2015) risolve entrambi i problemi aggiungendo due ingredienti al Q-learning con reti neurali.

**Buffer di replay dell'esperienza (experience replay).** Invece di addestrarsi con la transizione appena vissuta dall'agente e poi scartarla, salva ogni transizione `(s, a, r, s')` in una memoria e, a ogni passo di addestramento, campiona a caso un mini-batch da quella memoria. Questo rompe la correlazione tra passi consecutivi, riutilizza ogni transizione più volte e rende più uniforme la distribuzione dei dati di addestramento, perché combina esperienze raccolte con politiche leggermente diverse nel corso dell'addestramento.

**Rete target (target network).** Mantieni una seconda copia della rete, con i propri parametri θ⁻, e la usi solo per calcolare l'obiettivo di addestramento. Questa rete non viene aggiornata a ogni passo: viene sincronizzata con la rete principale ogni tot passi fissati, copiando i suoi pesi, e nel frattempo resta congelata. Così l'obiettivo smette di muoversi a ogni passo e diventa un riferimento stabile per un tratto di addestramento.

Con entrambi gli elementi, la funzione di perdita di DQN si scrive in modo intuitivo così:

`L(θ) = E[ (r + γ · max_a' Q(s',a';θ⁻) − Q(s,a;θ))² ]`

dove il valore atteso è calcolato sui mini-batch campionati dal buffer, θ sono i parametri della rete principale (quella che si addestra) e θ⁻ quelli della rete target (congelata tra una sincronizzazione e l'altra). Un dettaglio tecnico importante: nel calcolare il gradiente applichi uno "stop-gradient" sul termine dell'obiettivo, in modo che l'aggiornamento dei pesi fluisca solo verso θ, mai verso θ⁻. Senza questo dettaglio, staresti spostando lo stesso obiettivo per avvicinarlo alla tua previsione, invece del contrario.

La rete target resta un componente standard nella stragrande maggioranza delle implementazioni di DQN, insieme a un fattore di sconto tipico vicino a 0,99 e a un'esplorazione epsilon-greedy con decadimento (questo secondo punto lo metterai alla prova nella pratica di oggi).

**Tre miglioramenti successivi**, che implementerai nella lezione dieci:

- **Double DQN** separa la scelta della migliore azione successiva (con la rete principale) dalla sua valutazione (con la rete target), per ridurre la distorsione di sovrastima tipica del Q-learning
- **Dueling DQN** divide la rete in due rami —uno che stima il valore dello stato e uno che stima il vantaggio di ogni azione— e li combina alla fine
- **Prioritized Experience Replay** campiona il buffer dando più peso alle transizioni con maggiore errore di previsione, invece di campionare tutte allo stesso modo


## 4. Codice dal vivo: DQN su CartPole

*Traccia della lezione: costruisci l'esempio cella per cella, senza anticipare codice. Prima di eseguire il ciclo di addestramento, chiedi alla classe come sarà la curva della ricompensa.*

Addestrerai un DQN su CartPole, l'ambiente classico in cui un carrello deve mantenere in equilibrio un pendolo spostandosi a sinistra o a destra. Lo stato sono quattro numeri (posizione e velocità del carrello, angolo e velocità angolare del pendolo) e le azioni possibili sono due. È molto più semplice di Atari, ma contiene esattamente gli stessi ingredienti: rete Q, buffer di replay e rete target.

**Nota di progetto per oggi:** perché tu possa seguire il codice dal vivo senza distrazioni, questa lezione usa un epsilon **fisso** per tutto l'addestramento (non decresce). Nella pratica confronterai questa versione con una che incorpora il decadimento di epsilon, come raccomanda il manuale.

L'unica dipendenza esterna di questa lezione è `gymnasium`, la libreria che fornisce l'ambiente CartPole. PyTorch, `numpy` e `matplotlib` sono già preinstallati in Google Colab.


In [ ]:
# Instalamos gymnasium: es la UNICA dependencia externa de esta clase.
# PyTorch, numpy y matplotlib ya vienen preinstalados en Google Colab.
!pip install gymnasium


### Importazioni e seme

Fissiamo un seme perché l'esperimento sia, per quanto possibile, riproducibile.


In [ ]:
import random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import gymnasium as gym

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)


### Iperparametri

Scelti per addestrarsi rapidamente su una CPU normale di Colab, privilegiando la chiarezza rispetto alle prestazioni massime. `EPSILON` resta fisso per tutto l'addestramento: è la semplificazione di questa lezione che metterai in discussione nella pratica.


In [ ]:
NOMBRE_ENTORNO = "CartPole-v1"
NUM_EPISODIOS = 200                    # episodios de entrenamiento
GAMMA = 0.99                           # factor de descuento
TASA_APRENDIZAJE = 1e-3
TAMANO_BUFFER = 10000                  # capacidad del buffer de repeticion
TAMANO_LOTE = 64                       # tamano del mini-lote muestreado en cada paso
EPSILON = 0.1                          # exploracion epsilon-greedy, FIJA en esta clase
PASOS_ACTUALIZAR_RED_OBJETIVO = 500    # cada cuantos pasos se sincroniza la red objetivo
PASOS_MINIMOS_ANTES_ENTRENAR = 1000    # esperamos a tener algo de experiencia antes de entrenar


### La rete Q: l'approssimatore di funzioni

Riceve uno stato (quattro numeri, in CartPole) e restituisce un valore stimato per ogni azione possibile. È un piccolo perceptron multistrato, sufficiente per uno stato così breve.


In [ ]:
class RedQ(nn.Module):
    """Aproximador de funciones: recibe un estado y devuelve un valor
    estimado para cada accion posible."""

    def __init__(self, dim_estado, num_acciones):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(dim_estado, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_acciones),
        )

    def forward(self, estado):
        return self.capas(estado)


### Il buffer di replay dell'esperienza

Memorizza le transizioni passate e permette di campionare mini-batch a caso invece di addestrarsi solo con la transizione appena vissuta. `Transicion` è una tupla con nome, così il codice resta leggibile quando si costruiscono i mini-batch.


In [ ]:
Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferRepeticion:
    """Rompe la correlacion entre pasos consecutivos y hace que los datos de
    entrenamiento se parezcan mas a un conjunto i.i.d. (ver bloque 2 de esta clase)."""

    def __init__(self, capacidad):
        # deque con tamano maximo: al llenarse, descarta la transicion mas
        # antigua para dejar sitio a la nueva (comportamiento tipo FIFO)
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


### Politica epsilon-greedy

Con probabilità epsilon, sceglie un'azione a caso (esplorazione); altrimenti sceglie l'azione con il valore Q più alto secondo la rete online (sfruttamento). È la stessa politica che conosci già dalle lezioni precedenti.


In [ ]:
def elegir_accion(red_online, estado, epsilon, num_acciones):
    """Politica epsilon-greedy sobre los valores Q que estima la red online."""
    if random.random() < epsilon:
        return random.randrange(num_acciones)
    with torch.no_grad():
        estado_tensor = torch.tensor(estado, dtype=torch.float32).unsqueeze(0)
        valores_q = red_online(estado_tensor)
        return int(torch.argmax(valores_q, dim=1).item())


### Il passo di addestramento: qui vive la perdita di DQN

*Traccia della lezione: fermati qui e indica riga per riga la corrispondenza con la formula del blocco tre: `Q(s,a;θ)` è `valores_q`, `θ⁻` è `red_objetivo`, e `torch.no_grad()` è lo "stop-gradient".*

Campiona un mini-batch dal buffer e aggiusta la rete online per avvicinare la sua previsione all'obiettivo, calcolato con la rete target (target network).


In [ ]:
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Un paso de entrenamiento de DQN: muestrea un mini-lote del buffer y
    ajusta la red online para acercar su prediccion al objetivo calculado
    con la red objetivo (target network)."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    # Q(s,a;theta): valor que la red online asigna a la accion que de verdad se tomo
    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    # Objetivo: r + gamma * max_a' Q(s',a';theta-), calculado con la red objetivo.
    # torch.no_grad() implementa el "stop-gradient": el error se propaga
    # solo hacia la red online, nunca hacia la red objetivo.
    with torch.no_grad():
        max_q_siguiente = red_objetivo(siguientes_estados).max(dim=1)[0]
        objetivo = recompensas + GAMMA * max_q_siguiente * (1 - terminados)

    perdida = nn.functional.mse_loss(valores_q, objetivo)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


### Prepariamo ambiente, reti, buffer e ottimizzatore

Creiamo due copie di `RedQ`: `red_online` (quella che si addestra, θ) e `red_objetivo` (si congela tra le sincronizzazioni, θ⁻). All'inizio entrambe condividono gli stessi pesi.


In [ ]:
entorno = gym.make(NOMBRE_ENTORNO)
dim_estado = entorno.observation_space.shape[0]
num_acciones = entorno.action_space.n

red_online = RedQ(dim_estado, num_acciones)     # red principal: la que se entrena (theta)
red_objetivo = RedQ(dim_estado, num_acciones)   # red objetivo: congelada entre sincronizaciones (theta-)
red_objetivo.load_state_dict(red_online.state_dict())  # al empezar, theta- = theta
red_objetivo.eval()

optimizador = optim.Adam(red_online.parameters(), lr=TASA_APRENDIZAJE)
buffer = BufferRepeticion(TAMANO_BUFFER)

print("Dimension del estado:", dim_estado, "| Numero de acciones:", num_acciones)


### Il ciclo di addestramento

**Esegui la cella successiva.** Impiega solo alcuni secondi su una CPU normale. Osserva tre cose mentre gira:

- Si addestra solo quando il buffer ha accumulato abbastanza esperienza (`PASOS_MINIMOS_ANTES_ENTRENAR`)
- La rete target si sincronizza ogni `PASOS_ACTUALIZAR_RED_OBJETIVO` passi, non a ogni passo
- `EPSILON` resta fisso a 0,1 per tutta l'esecuzione: nella pratica di oggi cambierai questo valore


In [ ]:
recompensas_por_episodio = []
pasos_totales = 0

for episodio in range(NUM_EPISODIOS):
    estado, _ = entorno.reset(seed=SEMILLA + episodio)
    recompensa_episodio = 0.0
    terminado = False
    truncado = False

    while not (terminado or truncado):
        accion = elegir_accion(red_online, estado, EPSILON, num_acciones)
        siguiente_estado, recompensa, terminado, truncado, _ = entorno.step(accion)

        buffer.guardar(estado, accion, recompensa, siguiente_estado, terminado)
        estado = siguiente_estado
        recompensa_episodio += recompensa
        pasos_totales += 1

        # Solo entrenamos cuando el buffer ya acumulo suficiente experiencia
        if len(buffer) >= PASOS_MINIMOS_ANTES_ENTRENAR:
            entrenar_paso(red_online, red_objetivo, buffer, optimizador)

        # Sincronizamos la red objetivo cada cierto numero de pasos, no en cada paso
        if pasos_totales % PASOS_ACTUALIZAR_RED_OBJETIVO == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    recompensas_por_episodio.append(recompensa_episodio)

    if (episodio + 1) % 20 == 0:
        media_reciente = np.mean(recompensas_por_episodio[-20:])
        print(
            f"Episodio {episodio + 1}/{NUM_EPISODIOS} | epsilon={EPSILON:.2f} "
            f"| recompensa media (ultimos 20)={media_reciente:.1f}"
        )

entorno.close()


### Disegniamo la ricompensa per episodio

**Esegui questa cella per vedere la curva completa.**


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(recompensas_por_episodio, alpha=0.4, label="Recompensa por episodio")
# Media movil de 20 episodios, para ver la tendencia sin el ruido episodio a episodio
media_movil = np.convolve(recompensas_por_episodio, np.ones(20) / 20, mode="valid")
plt.plot(range(19, len(recompensas_por_episodio)), media_movil, label="Media movil (20 episodios)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("DQN en CartPole-v1: recompensa por episodio (epsilon fijo = 0.1)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Lettura del risultato**

*Traccia della lezione: fermati su questo grafico e chiedi alla classe quale tratto della curva corrisponde a "la rete non ha ancora imparato nulla".*

all'inizio la ricompensa per episodio è bassa e irregolare: il pendolo cade quasi subito, perché la rete ha visto pochissima esperienza. Man mano che il buffer si riempie e la rete si addestra, la media mobile dovrebbe salire, anche se con oscillazioni: questo mix di progresso e rumore episodio dopo episodio è normale in DQN, anche quando buffer di replay e rete target funzionano bene.


## Punti chiave

- I metodi tabulari falliscono quando il numero di stati cresce troppo: non stanno in memoria e l'agente raramente ripete lo stesso stato
- L'approssimazione di funzioni sostituisce la tabella con una funzione parametrizzata, `Q(s,a;θ)`, capace di generalizzare a stati nuovi
- Una rete neurale è, in sostanza, una funzione con pesi regolabili che si addestra per somigliare sempre di più alla funzione che vuoi approssimare
- Unire il Q-learning con una rete neurale in modo ingenuo è instabile, perché i dati dell'RL non sono i.i.d. e l'obiettivo di addestramento si sposta insieme alla rete che lo insegue (bootstrapping)
- Il buffer di replay dell'esperienza rompe la correlazione tra passi consecutivi e riutilizza ogni transizione più volte
- La rete target congela una copia dei parametri per un tratto di addestramento: così l'obiettivo smette di muoversi a ogni passo e l'addestramento si stabilizza
- Double DQN, Dueling DQN e Prioritized Experience Replay sono miglioramenti su DQN che implementerai nella lezione dieci

## Mettilo in pratica

Prendi il codice di questa lezione e modificalo tu stesso:

- **Cambia `PASOS_ACTUALIZAR_RED_OBJETIVO`** (prova con 50, con 500 e con 5.000) e osserva cosa succede alla stabilità della curva: una rete target che si sincronizza troppo in fretta somiglia sempre di più al Q-learning ingenuo e instabile del blocco due
- **Cambia `TAMANO_BUFFER`** (prova con 500 e con 50.000) e verifica se un buffer piccolo rende l'addestramento più erratico, perdendo diversità nelle esperienze passate
- **Elimina temporaneamente la rete target** (usa `red_online` al posto di `red_objetivo` dentro `entrenar_paso`) e confronta la curva risultante con la versione originale

## Prima della prossima lezione

**Apri il notebook di pratica di questa lezione.** Completerai tu stesso la perdita di DQN da zero e programmerai un decadimento di epsilon per confrontarlo con l'epsilon fisso usato oggi.

DQN impara valori di azione e ne deriva la politica in modo indiretto, scegliendo sempre l'azione con Q più alto. Nella lezione otto conoscerai un'altra famiglia di metodi, i gradienti di politica e i metodi actor-critic, che imparano la politica in modo diretto, senza passare da una funzione di valore intermedia.
